
# Kuroshio Daily R1 FIX5
## Event-Triggered Top-K Ridge Escape
### 原 characteristic-line 方法的小改进版（无 CNN、无新训练）

本轮目标不是设计一个新的流轴模型，而是**自动化原论文中“陷入涡旋后回退并重新识别”的人工调参过程**。

正常情况下，FIX5 与原方法完全相同：

```text
辅助线
→ 加权平均流向
→ 新辅助线
→ 选择最大流速点
→ 沿流向继续推进
```

只有出现以下异常事件时才启动恢复：**路径返回旧轨迹（loop/repeat）**，或原方法在当前位置出现 **no_valid_aux_flow / no_candidate** 而提前中断：

```text
rollback
→ 20/30/40 km 自适应前跳
→ 当前副线上 Top-K 高流速脊候选
→ 短程 characteristic-line 预演
→ 排除重新返回历史路径的候选
→ 选择能够保持高速并持续向下游推进的分支
```

### 为什么比 CNN 更适合这次论文大修？

- 不改变“黑潮轴 = 速度主脊”的物理定义；
- 不需要标签、不需要额外神经网络训练；
- 正常日期与原方法一致；
- 只在原方法已经判定“陷涡”的时刻介入；
- 本质上只是把原来的“回退 + 固定20 km + ±30°”变成自动的多候选恢复；
- 非常适合 500+ 个日平均场批量自动处理，减少人工逐日重跑。

### 默认运行策略

本 Notebook 会先在 difficult + normal QA 样本上验证。
若 Gate 自动通过，默认继续跑全量 2023-09~2025-03，并重新输出日尺度双分支数据集。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, json, math, random
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"

    CACHE50_DIR: str = "cache_50m"
    BASELINE_DIR: str = "tracks_baseline_FIX2_east0"

    MANIFEST_FILE: str = "reports/daily_manifest.csv"
    BASELINE_REPORT: str = "reports/baseline_tracker_report_FIX2.csv"
    MANUAL_QC_TEMPLATE: str = "reports/manual_qc_template.csv"

    OUT_ROOT: str = "R1_FIX5"
    TRACK_DIR: str = "R1_FIX5/tracks"
    REPORT_DIR: str = "R1_FIX5/reports"
    FIGURE_DIR: str = "R1_FIX5/figures/manual_qc"
    DATASET_DIR: str = "R1_FIX5/dataset"

    # original characteristic-line
    SEED_LON_MIN: float = 121.0
    SEED_LON_MAX: float = 122.0
    SEED_LAT_MIN: float = 22.0
    SEED_LAT_MAX: float = 23.0
    TERMINAL_LON: float = 146.0
    EXTENSION_LON: float = 141.0

    AUX_HALF_KM: float = 70.0
    AUX_POINTS: int = 21

    STEP_MAIN_KM: float = 10.0
    STEP_EXTENSION_KM: float = 50.0
    MAX_TRACK_POINTS: int = 1000

    REPEAT_THRESHOLD_KM: float = 1.5
    REPEAT_SKIP_RECENT: int = 5

    # FIX5 escape
    ESCAPE_STEP_OPTIONS_KM: tuple = (20.0, 30.0, 40.0)
    ESCAPE_TOP_K: int = 5
    ESCAPE_SPEED_FLOOR: float = 0.82
    ESCAPE_GEOM_ANGLE_DEG: float = 45.0

    PREVIEW_STEPS: int = 7
    PREVIEW_STEP_MAIN_KM: float = 10.0
    PREVIEW_HISTORY_GUARD_KM: float = 22.0
    PREVIEW_SKIP_RECENT: int = 10

    # preview score
    W_MEAN_SPEED: float = 0.42
    W_MIN_SPEED: float = 0.18
    W_PROGRESS: float = 0.22
    W_DIRECTION: float = 0.10
    W_STRAIGHTNESS: float = 0.08

    HISTORY_RETURN_PENALTY: float = 2.0
    PREVIEW_LOOP_PENALTY: float = 2.0

    MAX_ESCAPE_EVENTS: int = 6

    # QA
    QA_RANDOM_GOOD_ZERO_LOOP: int = 24
    QA_RANDOM_COMPLETED_WITH_LOOP: int = 24
    QA_RANDOM_FAILED: int = 32
    RANDOM_SEED: int = 20260925

    # QA gate
    MAX_ZERO_LOOP_REGRESSION: int = 0
    MAX_COMPLETED_REGRESSION: int = 1
    MIN_FAIL_RECOVERY: int = 4
    MAX_PATH_TURN_DEGRADATION_DEG: float = 6.0

    # 为了节省重投时间：QA通过后自动跑全量
    AUTO_RUN_FULL_IF_GATE: bool = True

cfg = CFG()

ROOT = Path(cfg.PROJECT_ROOT)
CACHE50 = ROOT / cfg.CACHE50_DIR
BASE_DIR = ROOT / cfg.BASELINE_DIR
MANIFEST_PATH = ROOT / cfg.MANIFEST_FILE
BASE_REPORT_PATH = ROOT / cfg.BASELINE_REPORT
MANUAL_QC_PATH = ROOT / cfg.MANUAL_QC_TEMPLATE

OUT_ROOT = ROOT / cfg.OUT_ROOT
TRACK_DIR = ROOT / cfg.TRACK_DIR
REPORT_DIR = ROOT / cfg.REPORT_DIR
FIGURE_DIR = ROOT / cfg.FIGURE_DIR
DATASET_DIR = ROOT / cfg.DATASET_DIR

for p in [OUT_ROOT, TRACK_DIR, REPORT_DIR, FIGURE_DIR, DATASET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

# ===== 1. 输入检查 =====
for p in [CACHE50, BASE_DIR, MANIFEST_PATH, BASE_REPORT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"missing: {p}")

manifest = pd.read_csv(MANIFEST_PATH, parse_dates=["date"])
baseline_report = pd.read_csv(BASE_REPORT_PATH, parse_dates=["date"])

print("manifest:", len(manifest))
print("baseline days:", len(baseline_report))
print("baseline completion:", baseline_report["completed"].mean())
print("baseline loop-day:", (baseline_report["loop_events"].fillna(0)>0).mean())


In [ ]:

# ===== 2. 几何与插值 =====
from scipy.interpolate import RegularGridInterpolator
from scipy.signal import find_peaks

EARTH_R_KM = 6371.0088

def wrap_pi(a):
    return (a + np.pi) % (2*np.pi) - np.pi

def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.asarray, (lon1,lat1,lon2,lat2))
    p1,p2 = np.deg2rad(lat1),np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    dp = p2-p1
    aa = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*EARTH_R_KM*np.arcsin(np.minimum(1.0,np.sqrt(aa)))

def angle_east_rad(lon1,lat1,lon2,lat2):
    p1,p2 = np.deg2rad(lat1),np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    yy = np.sin(dl)*np.cos(p2)
    xx = np.cos(p1)*np.sin(p2)-np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav = np.arctan2(yy,xx)
    return wrap_pi(np.pi/2-nav)

def forward_point(lon,lat,east_angle,distance_km):
    nav = np.pi/2-east_angle
    phi1,lam1 = np.deg2rad(lat),np.deg2rad(lon)
    d = distance_km/EARTH_R_KM
    phi2 = np.arcsin(
        np.sin(phi1)*np.cos(d) +
        np.cos(phi1)*np.sin(d)*np.cos(nav)
    )
    lam2 = lam1 + np.arctan2(
        np.sin(nav)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)),float(np.rad2deg(phi2))

def aux_line(lon,lat,flow_dir,half_km=None,n=None):
    half_km = cfg.AUX_HALF_KM if half_km is None else half_km
    n = cfg.AUX_POINTS if n is None else n
    ds = np.linspace(-half_km,half_km,n)
    xs,ys = [],[]
    for s in ds:
        b = flow_dir + (np.pi/2 if s>=0 else -np.pi/2)
        x,y = forward_point(lon,lat,b,abs(float(s)))
        xs.append(x); ys.append(y)
    return np.asarray(xs),np.asarray(ys),ds

def matlab_aux_weights(n):
    k=np.arange(1,n+1,dtype=float)
    return 1.0-np.abs((n+1)/2.0-k)/(n-1)

def make_interpolators(lon,lat,u,v):
    return (
        RegularGridInterpolator((lat,lon),u,bounds_error=False,fill_value=np.nan),
        RegularGridInterpolator((lat,lon),v,bounds_error=False,fill_value=np.nan)
    )

def interp_uv(ui,vi,lons,lats):
    pts=np.column_stack([lats,lons])
    return ui(pts),vi(pts)

def interp_one(ui,vi,lon,lat):
    uu=float(ui([[lat,lon]])[0])
    vv=float(vi([[lat,lon]])[0])
    return uu,vv

# geometry smoke
xe,ye=forward_point(130,30,0,10)
xn,yn=forward_point(130,30,np.pi/2,10)
assert xe>130 and abs(ye-30)<0.05
assert yn>30 and abs(xn-130)<0.05
print("[geometry smoke] PASS")


In [ ]:

# ===== 3. 原方法单步：正常状态完全保持最大流速规则 =====
def weighted_mean_direction(ui,vi,center_lon,center_lat,current_dir):
    xl,yl,_ = aux_line(center_lon,center_lat,current_dir)
    uu,vv = interp_uv(ui,vi,xl,yl)

    valid=np.isfinite(uu)&np.isfinite(vv)
    if not valid.any():
        return None

    w=matlab_aux_weights(len(xl))
    den=np.sum(w[valid])
    if den<=0:
        return None

    mean_u=np.sum(uu[valid]*w[valid])/den
    mean_v=np.sum(vv[valid]*w[valid])/den
    return float(np.arctan2(mean_v,mean_u))

def original_max_speed_candidate(ui,vi,center_lon,center_lat,mean_dir):
    xl,yl,_ = aux_line(center_lon,center_lat,mean_dir)
    uu,vv = interp_uv(ui,vi,xl,yl)
    speed=np.hypot(uu,vv)

    valid=np.isfinite(speed)
    if not valid.any():
        return None

    k=int(np.nanargmax(speed))
    return {
        "lon":float(xl[k]),
        "lat":float(yl[k]),
        "u":float(uu[k]),
        "v":float(vv[k]),
        "speed":float(speed[k]),
        "speed_ratio":1.0,
    }

def repeat_index(path_lon,path_lat,lon,lat):
    if len(path_lon) <= 8:
        return None

    hx=np.asarray(path_lon[:-cfg.REPEAT_SKIP_RECENT])
    hy=np.asarray(path_lat[:-cfg.REPEAT_SKIP_RECENT])
    if len(hx)==0:
        return None

    dd=haversine_km(hx,hy,lon,lat)
    if np.isfinite(dd).any() and np.nanmin(dd)<cfg.REPEAT_THRESHOLD_KM:
        return int(np.nanargmin(dd))
    return None


In [ ]:

# ===== 4. FIX5核心：事件触发 Top-K 高流速脊短程预演 =====
def topk_ridge_candidates(
    ui,vi,
    center_lon,center_lat,
    mean_dir,
    anchor_lon,anchor_lat
):
    xl,yl,_ = aux_line(center_lon,center_lat,mean_dir)
    uu,vv = interp_uv(ui,vi,xl,yl)
    speed=np.hypot(uu,vv)

    valid=np.isfinite(speed)&np.isfinite(uu)&np.isfinite(vv)
    if not valid.any():
        return []

    smax=float(np.nanmax(speed[valid]))
    if smax<=0:
        return []

    # 局地峰 + 全局top speed
    ss=np.where(valid,speed,-1e9)
    peaks,_=find_peaks(ss)
    ids=set(int(p) for p in peaks if valid[p])

    good=np.where(valid)[0]
    top=good[np.argsort(speed[good])[::-1][:max(cfg.ESCAPE_TOP_K*2, cfg.ESCAPE_TOP_K)]]
    ids.update(map(int,top))

    out=[]
    for k in ids:
        ratio=float(speed[k]/(smax+1e-8))
        if ratio<cfg.ESCAPE_SPEED_FLOOR:
            continue

        geom=angle_east_rad(
            anchor_lon,anchor_lat,
            float(xl[k]),float(yl[k])
        )
        diff=abs(np.rad2deg(wrap_pi(geom-mean_dir)))

        if diff>cfg.ESCAPE_GEOM_ANGLE_DEG:
            continue

        out.append({
            "k":int(k),
            "lon":float(xl[k]),
            "lat":float(yl[k]),
            "u":float(uu[k]),
            "v":float(vv[k]),
            "speed":float(speed[k]),
            "speed_ratio":ratio,
            "geom_diff_deg":float(diff),
        })

    out=sorted(
        out,
        key=lambda d:d["speed"],
        reverse=True
    )

    return out[:cfg.ESCAPE_TOP_K]

def preview_candidate(
    ui,vi,
    candidate,
    mean_dir0,
    history_lon,history_lat
):
    """
    从候选点开始，用“原最大流速 characteristic-line”预演 PREVIEW_STEPS。
    仅用于选择恢复分支，不改变正常追踪定义。
    """
    x=float(candidate["lon"])
    y=float(candidate["lat"])
    current_dir=float(mean_dir0)

    speeds=[float(candidate["speed"])]
    speed_ratios=[float(candidate["speed_ratio"])]
    preview_lon=[x]
    preview_lat=[y]
    directions=[current_dir]

    history_return=False
    preview_loop=False

    for step in range(cfg.PREVIEW_STEPS):
        # 历史回访
        if len(history_lon)>cfg.PREVIEW_SKIP_RECENT:
            hx=np.asarray(history_lon[:-cfg.PREVIEW_SKIP_RECENT])
            hy=np.asarray(history_lat[:-cfg.PREVIEW_SKIP_RECENT])
            dd=haversine_km(hx,hy,x,y)
            if np.isfinite(dd).any() and np.nanmin(dd)<cfg.PREVIEW_HISTORY_GUARD_KM:
                history_return=True

        # preview自身闭环
        if len(preview_lon)>5:
            dd2=haversine_km(
                np.asarray(preview_lon[:-4]),
                np.asarray(preview_lat[:-4]),
                x,y
            )
            if np.isfinite(dd2).any() and np.nanmin(dd2)<cfg.PREVIEW_HISTORY_GUARD_KM:
                preview_loop=True

        md=weighted_mean_direction(
            ui,vi,x,y,current_dir
        )
        if md is None:
            break

        cand=original_max_speed_candidate(
            ui,vi,x,y,md
        )
        if cand is None:
            break

        # 估算该点相对副线最大值，原candidate本身就是max所以=1
        speeds.append(cand["speed"])
        speed_ratios.append(1.0)

        x=float(cand["lon"])
        y=float(cand["lat"])
        current_dir=float(md)

        preview_lon.append(x)
        preview_lat.append(y)
        directions.append(current_dir)

        step_km=(
            cfg.STEP_EXTENSION_KM
            if x>cfg.EXTENSION_LON
            else cfg.PREVIEW_STEP_MAIN_KM
        )
        x,y=forward_point(
            x,y,current_dir,step_km
        )

    if len(preview_lon)<2:
        return {
            **candidate,
            "score":-999.0,
            "mean_speed_ratio":candidate["speed_ratio"],
            "min_speed_ratio":candidate["speed_ratio"],
            "progress":0.0,
            "direction_score":0.0,
            "straightness":0.0,
            "history_return":True,
            "preview_loop":True,
            "preview_n":len(preview_lon),
        }

    # 速度归一化：以起始候选所在副线max为基准
    s0=max(float(candidate["speed"]),1e-8)
    sr=np.clip(np.asarray(speeds)/s0,0,1.25)
    mean_speed=float(np.clip(np.mean(sr),0,1))
    min_speed=float(np.clip(np.min(sr),0,1))

    # 短程净下游投影
    net_dist=float(
        haversine_km(
            preview_lon[0],preview_lat[0],
            preview_lon[-1],preview_lat[-1]
        )
    )
    net_dir=float(
        angle_east_rad(
            preview_lon[0],preview_lat[0],
            preview_lon[-1],preview_lat[-1]
        )
    )
    theoretical=max(
        1e-6,
        cfg.PREVIEW_STEP_MAIN_KM*(len(preview_lon)-1)
    )
    progress=float(np.clip(
        max(0.0,net_dist*np.cos(wrap_pi(net_dir-mean_dir0))) / theoretical,
        0.0,1.0
    ))

    dirs=np.asarray(directions)
    direction_score=float(np.clip(
        np.mean(
            (np.cos(wrap_pi(dirs-mean_dir0))+1)/2
        ),
        0,1
    ))

    # straightness = 净位移 / 实际路径长度
    plen=float(np.sum(
        haversine_km(
            np.asarray(preview_lon[:-1]),
            np.asarray(preview_lat[:-1]),
            np.asarray(preview_lon[1:]),
            np.asarray(preview_lat[1:])
        )
    ))
    straightness=float(np.clip(
        net_dist/(plen+1e-8),
        0,1
    ))

    score=(
        cfg.W_MEAN_SPEED*mean_speed +
        cfg.W_MIN_SPEED*min_speed +
        cfg.W_PROGRESS*progress +
        cfg.W_DIRECTION*direction_score +
        cfg.W_STRAIGHTNESS*straightness -
        cfg.HISTORY_RETURN_PENALTY*float(history_return) -
        cfg.PREVIEW_LOOP_PENALTY*float(preview_loop)
    )

    return {
        **candidate,
        "score":float(score),
        "mean_speed_ratio":mean_speed,
        "min_speed_ratio":min_speed,
        "progress":progress,
        "direction_score":direction_score,
        "straightness":straightness,
        "history_return":bool(history_return),
        "preview_loop":bool(preview_loop),
        "preview_n":int(len(preview_lon)),
    }

def adaptive_escape(
    ui,vi,
    anchor_lon,anchor_lat,
    anchor_dir,
    history_lon,history_lat
):
    all_eval=[]

    for jump_km in cfg.ESCAPE_STEP_OPTIONS_KM:
        center_lon,center_lat=forward_point(
            anchor_lon,anchor_lat,
            anchor_dir,
            jump_km
        )

        md=weighted_mean_direction(
            ui,vi,
            center_lon,center_lat,
            anchor_dir
        )
        if md is None:
            continue

        cands=topk_ridge_candidates(
            ui,vi,
            center_lon,center_lat,
            md,
            anchor_lon,anchor_lat
        )

        for c in cands:
            e=preview_candidate(
                ui,vi,
                c,md,
                history_lon,history_lat
            )
            e["jump_km"]=float(jump_km)
            e["mean_dir"]=float(md)
            all_eval.append(e)

        # 若本级已有明确无回访、无preview loop的候选，
        # 优先不继续扩大jump，保持最小改动。
        clean=[
            e for e in all_eval
            if (
                not e["history_return"] and
                not e["preview_loop"] and
                e["progress"]>=0.35
            )
        ]
        if clean:
            break

    if not all_eval:
        return None, []

    clean=[
        e for e in all_eval
        if (
            not e["history_return"] and
            not e["preview_loop"]
        )
    ]

    pool=clean if clean else all_eval
    best=max(pool,key=lambda d:d["score"])

    return best,all_eval


In [ ]:

# ===== 5. FIX5 tracker：仅在异常事件时介入 =====
def _seed_point(lon,lat,u,v):
    ix=np.where(
        (lon>=cfg.SEED_LON_MIN)&
        (lon<=cfg.SEED_LON_MAX)
    )[0]
    iy=np.where(
        (lat>=cfg.SEED_LAT_MIN)&
        (lat<=cfg.SEED_LAT_MAX)
    )[0]

    su=u[np.ix_(iy,ix)]
    sv=v[np.ix_(iy,ix)]
    sp2=su**2+sv**2

    if not np.isfinite(sp2).any():
        raise RuntimeError("seed region invalid")

    j,i=np.unravel_index(np.nanargmax(sp2),sp2.shape)
    return (
        float(lon[ix[i]]),
        float(lat[iy[j]]),
        float(np.arctan2(sv[j,i],su[j,i]))
    )

def _apply_escape(
    ui,vi,
    path_lon,path_lat,dirs,
    escape_events,escape_rows,
    trigger_name
):
    """
    从最后稳定流轴点启动自适应恢复。
    返回:
      success, new_center_lon, new_center_lat, new_dir,
      escape_events, escape_success_increment
    """
    if len(path_lon)<2 or len(dirs)<1:
        return False, None, None, None, escape_events, 0

    escape_events += 1
    if escape_events > cfg.MAX_ESCAPE_EVENTS:
        return False, None, None, None, escape_events, 0

    anchor_lon=float(path_lon[-1])
    anchor_lat=float(path_lat[-1])
    anchor_dir=float(dirs[-1])

    best,evaluated=adaptive_escape(
        ui,vi,
        anchor_lon,anchor_lat,
        anchor_dir,
        path_lon,path_lat
    )

    if best is None:
        return False, None, None, None, escape_events, 0

    if (
        best["history_return"] or
        best["preview_loop"] or
        best["score"]<0
    ):
        return False, None, None, None, escape_events, 0

    path_lon.append(float(best["lon"]))
    path_lat.append(float(best["lat"]))
    dirs.append(float(best["mean_dir"]))

    escape_rows.append({
        "event":escape_events,
        "trigger":str(trigger_name),
        "jump_km":best["jump_km"],
        "candidate_speed_ratio":best["speed_ratio"],
        "mean_speed_ratio":best["mean_speed_ratio"],
        "min_speed_ratio":best["min_speed_ratio"],
        "progress":best["progress"],
        "direction_score":best["direction_score"],
        "straightness":best["straightness"],
        "score":best["score"],
        "n_candidates":len(evaluated),
    })

    step_km=(
        cfg.STEP_EXTENSION_KM
        if best["lon"]>cfg.EXTENSION_LON
        else cfg.STEP_MAIN_KM
    )

    new_center_lon,new_center_lat=forward_point(
        float(best["lon"]),
        float(best["lat"]),
        float(best["mean_dir"]),
        step_km
    )

    return (
        True,
        new_center_lon,
        new_center_lat,
        float(best["mean_dir"]),
        escape_events,
        1
    )

def track_fix5(lon,lat,u,v):
    ui,vi=make_interpolators(lon,lat,u,v)
    lon0,lat0,theta0=_seed_point(lon,lat,u,v)

    path_lon=[]
    path_lat=[]
    dirs=[]

    loop_events=0
    escape_events=0
    escape_success=0
    escape_rows=[]

    center_lon,center_lat=lon0,lat0
    current_dir=theta0
    stop_reason="max_points"

    for _ in range(cfg.MAX_TRACK_POINTS):
        md=weighted_mean_direction(
            ui,vi,
            center_lon,center_lat,
            current_dir
        )

        # 原方法在副线上完全找不到有效流场：
        # 不立刻判死，先从最后稳定点做一次自适应恢复
        if md is None:
            ok,nx,ny,ndir,escape_events,inc=_apply_escape(
                ui,vi,
                path_lon,path_lat,dirs,
                escape_events,escape_rows,
                "no_valid_aux_flow"
            )
            escape_success += inc
            if ok:
                center_lon,center_lat,current_dir=nx,ny,ndir
                continue

            stop_reason="no_valid_aux_flow"
            break

        cand=original_max_speed_candidate(
            ui,vi,
            center_lon,center_lat,
            md
        )

        # 同理：有平均方向但新副线没有可选点时，先自动恢复
        if cand is None:
            ok,nx,ny,ndir,escape_events,inc=_apply_escape(
                ui,vi,
                path_lon,path_lat,dirs,
                escape_events,escape_rows,
                "no_candidate"
            )
            escape_success += inc
            if ok:
                center_lon,center_lat,current_dir=nx,ny,ndir
                continue

            stop_reason="no_candidate"
            break

        cl,ca=cand["lon"],cand["lat"]

        ri=repeat_index(
            path_lon,path_lat,cl,ca
        )

        if ri is not None:
            loop_events += 1

            # rollback 与原方法一致
            path_lon=path_lon[:ri+1]
            path_lat=path_lat[:ri+1]
            dirs=dirs[:ri+1]

            ok,nx,ny,ndir,escape_events,inc=_apply_escape(
                ui,vi,
                path_lon,path_lat,dirs,
                escape_events,escape_rows,
                "repeat_loop"
            )
            escape_success += inc

            if not ok:
                stop_reason="escape_no_safe_branch"
                break

            center_lon,center_lat,current_dir=nx,ny,ndir
            continue

        # 正常点：与原方法完全一致
        path_lon.append(cl)
        path_lat.append(ca)
        dirs.append(md)

        if cl>=cfg.TERMINAL_LON:
            stop_reason="terminal_lon"
            break

        step_km=(
            cfg.STEP_EXTENSION_KM
            if cl>cfg.EXTENSION_LON
            else cfg.STEP_MAIN_KM
        )
        center_lon,center_lat=forward_point(
            cl,ca,md,step_km
        )
        current_dir=md

    x=np.asarray(path_lon,dtype=np.float32)
    y=np.asarray(path_lat,dtype=np.float32)

    completed=bool(
        len(x)>2 and
        np.isfinite(x).any() and
        np.nanmax(x)>=cfg.TERMINAL_LON
    )

    return {
        "lon":x,
        "lat":y,
        "direction":np.asarray(dirs,dtype=np.float32),
        "completed":completed,
        "loop_events":int(loop_events),
        "escape_events":int(escape_events),
        "escape_success":int(escape_success),
        "n_points":int(len(x)),
        "stop_reason":stop_reason,
        "escape_rows":escape_rows,
    }

def track_metrics(tr):
    x=np.asarray(tr["lon"])
    y=np.asarray(tr["lat"])

    out={
        "completed":bool(tr["completed"]),
        "loop_events":int(tr["loop_events"]),
        "escape_events":int(tr["escape_events"]),
        "escape_success":int(tr["escape_success"]),
        "n_points":int(tr["n_points"]),
        "stop_reason":tr["stop_reason"],
        "max_lon":float(np.nanmax(x)) if len(x) else np.nan,
        "final_lon":float(x[-1]) if len(x) else np.nan,
        "final_lat":float(y[-1]) if len(y) else np.nan,
    }

    if len(x)>1:
        out["path_km"]=float(np.sum(
            haversine_km(
                x[:-1],y[:-1],x[1:],y[1:]
            )
        ))
    else:
        out["path_km"]=0.0

    if len(x)>3:
        dd=np.array([
            angle_east_rad(
                x[i],y[i],
                x[i+1],y[i+1]
            )
            for i in range(len(x)-1)
        ])
        turns=np.abs(
            np.rad2deg(
                wrap_pi(np.diff(dd))
            )
        )
        out["p95_turn_deg"]=float(
            np.nanpercentile(turns,95)
        )
    else:
        out["p95_turn_deg"]=np.nan

    if tr["escape_rows"]:
        er=pd.DataFrame(tr["escape_rows"])
        out["escape_mean_speed_ratio"]=float(er["mean_speed_ratio"].mean())
        out["escape_min_speed_ratio"]=float(er["min_speed_ratio"].min())
        out["escape_mean_progress"]=float(er["progress"].mean())
        out["escape_mean_score"]=float(er["score"].mean())
        out["escape_repeat_events"]=int((er["trigger"]=="repeat_loop").sum())
        out["escape_no_candidate_events"]=int((er["trigger"]=="no_candidate").sum())
        out["escape_no_valid_events"]=int((er["trigger"]=="no_valid_aux_flow").sum())
    else:
        out["escape_mean_speed_ratio"]=np.nan
        out["escape_min_speed_ratio"]=np.nan
        out["escape_mean_progress"]=np.nan
        out["escape_mean_score"]=np.nan
        out["escape_repeat_events"]=0
        out["escape_no_candidate_events"]=0
        out["escape_no_valid_events"]=0

    return out

def save_track(tr,path):
    escape_json=json.dumps(
        tr["escape_rows"],
        ensure_ascii=False
    )
    np.savez_compressed(
        path,
        lon=tr["lon"],
        lat=tr["lat"],
        direction=tr["direction"],
        completed=np.array(tr["completed"]),
        loop_events=np.array(tr["loop_events"]),
        escape_events=np.array(tr["escape_events"]),
        escape_success=np.array(tr["escape_success"]),
        n_points=np.array(tr["n_points"]),
        stop_reason=np.array(tr["stop_reason"]),
        escape_json=np.array(escape_json),
        tracker_version=np.array("R1_FIX5_EVENT_TRIGGERED_TOPK_ESCAPE_V2")
    )


In [ ]:

# ===== 6. QA日期：重点困难样本 + 正常样本 =====
manual_dates=[]

if MANUAL_QC_PATH.exists():
    q=pd.read_csv(MANUAL_QC_PATH)
    if "date" in q.columns:
        manual_dates=list(
            pd.to_datetime(q["date"])
            .dropna()
            .dt.normalize()
            .unique()
        )

good0=baseline_report[
    baseline_report["completed"] &
    (baseline_report["loop_events"]==0)
]["date"].dropna().tolist()

good_loop=baseline_report[
    baseline_report["completed"] &
    (baseline_report["loop_events"]>0)
]["date"].dropna().tolist()

failed=baseline_report[
    ~baseline_report["completed"]
]["date"].dropna().tolist()

rng=np.random.default_rng(cfg.RANDOM_SEED)

def sample_list(lst,n):
    if not lst:
        return []
    return list(rng.choice(
        lst,
        size=min(n,len(lst)),
        replace=False
    ))

qa_dates=sorted(set(
    list(pd.to_datetime(manual_dates)) +
    list(pd.to_datetime(sample_list(good0,cfg.QA_RANDOM_GOOD_ZERO_LOOP))) +
    list(pd.to_datetime(sample_list(good_loop,cfg.QA_RANDOM_COMPLETED_WITH_LOOP))) +
    list(pd.to_datetime(sample_list(failed,cfg.QA_RANDOM_FAILED)))
))

qa_dates=[
    d for d in qa_dates
    if (CACHE50/f"{pd.Timestamp(d):%Y%m%d}.npz").exists()
]

print("QA dates:",len(qa_dates))

pd.DataFrame({"date":qa_dates}).to_csv(
    REPORT_DIR/"qa_dates_FIX5.csv",
    index=False
)


In [ ]:

# ===== 7. QA运行 =====
from tqdm.auto import tqdm

rows=[]

for dt in tqdm(qa_dates,desc="FIX5 QA"):
    dt=pd.Timestamp(dt)
    z=np.load(CACHE50/f"{dt:%Y%m%d}.npz")

    tr=track_fix5(
        z["lon"],z["lat"],z["u"],z["v"]
    )
    met=track_metrics(tr)
    met["date"]=dt
    rows.append(met)

    save_track(
        tr,
        TRACK_DIR/f"{dt:%Y%m%d}.npz"
    )

qa_fix5=pd.DataFrame(rows).sort_values("date")

cmp=baseline_report.merge(
    qa_fix5,
    on="date",
    suffixes=("_base","_fix5")
)

qa_fix5.to_csv(
    REPORT_DIR/"qa_fix5_report.csv",
    index=False,
    encoding="utf-8-sig"
)
cmp.to_csv(
    REPORT_DIR/"qa_baseline_vs_fix5.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    cmp[[
        "date",
        "completed_base","completed_fix5",
        "loop_events_base","loop_events_fix5",
        "escape_events",
        "max_lon_base","max_lon_fix5",
        "p95_turn_deg_base","p95_turn_deg_fix5"
    ]].head(30)
)


In [ ]:

# ===== 8. FIX5 Gate =====
zero_loop_regression=int(
    (
        cmp["completed_base"] &
        (cmp["loop_events_base"]==0) &
        (~cmp["completed_fix5"])
    ).sum()
)

completed_regression=int(
    (
        cmp["completed_base"] &
        (~cmp["completed_fix5"])
    ).sum()
)

fail_recovery=int(
    (
        (~cmp["completed_base"]) &
        cmp["completed_fix5"]
    ).sum()
)

completion_base=float(cmp["completed_base"].mean())
completion_fix5=float(cmp["completed_fix5"].mean())

loop_base=float((cmp["loop_events_base"]>0).mean())
loop_fix5=float((cmp["loop_events_fix5"]>0).mean())

turn_base=float(cmp["p95_turn_deg_base"].median())
turn_fix5=float(cmp["p95_turn_deg_fix5"].median())

# 对baseline 0-loop样本，正常情况下FIX5应完全相同
good0_cmp=cmp[
    cmp["completed_base"] &
    (cmp["loop_events_base"]==0)
].copy()

same_terminal_km=[]
for r in good0_cmp.itertuples(index=False):
    same_terminal_km.append(
        float(haversine_km(
            r.final_lon_base,r.final_lat_base,
            r.final_lon_fix5,r.final_lat_fix5
        ))
    )

max_good0_terminal_diff=max(same_terminal_km) if same_terminal_km else 0.0

checks={
    "zero_loop_regression":
        zero_loop_regression<=cfg.MAX_ZERO_LOOP_REGRESSION,
    "completed_regression":
        completed_regression<=cfg.MAX_COMPLETED_REGRESSION,
    "fail_recovery":
        fail_recovery>=cfg.MIN_FAIL_RECOVERY,
    "completion_not_worse":
        completion_fix5>=completion_base,
    "turn_not_materially_worse":
        turn_fix5<=turn_base+cfg.MAX_PATH_TURN_DEGRADATION_DEG,
    "zero_loop_path_invariance":
        max_good0_terminal_diff<1.0,
}

AUTO_GATE_PASS=all(checks.values())

summary={
    "qa_days":int(len(cmp)),
    "completion_base":completion_base,
    "completion_fix5":completion_fix5,
    "loop_day_base":loop_base,
    "loop_day_fix5":loop_fix5,
    "zero_loop_regression":zero_loop_regression,
    "completed_regression":completed_regression,
    "fail_recovery":fail_recovery,
    "median_p95_turn_base":turn_base,
    "median_p95_turn_fix5":turn_fix5,
    "max_zero_loop_terminal_diff_km":max_good0_terminal_diff,
    "checks":checks,
    "auto_gate_pass":AUTO_GATE_PASS,
}

print(json.dumps(summary,indent=2,ensure_ascii=False))

with open(
    REPORT_DIR/"qa_gate_FIX5.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,f,
        indent=2,
        ensure_ascii=False
    )


In [ ]:

# ===== 9. QA图：只比较原baseline与FIX5，不再画CNN =====
import matplotlib.pyplot as plt

def load_xy(path):
    z=np.load(path)
    return z["lon"],z["lat"]

# manual dates + 最大恢复/失败日期
priority=cmp.copy()
priority["priority"]=(
    (~priority["completed_fix5"]).astype(float)*5 +
    priority["loop_events_fix5"].fillna(0)*2 +
    np.maximum(
        0,
        priority["p95_turn_deg_fix5"].fillna(180)-90
    )/30
)

plot_dates=sorted(set(
    list(pd.to_datetime(manual_dates)) +
    priority.sort_values(
        "priority",
        ascending=False
    ).head(24)["date"].tolist()
))

for dt in plot_dates:
    dt=pd.Timestamp(dt)

    cache=CACHE50/f"{dt:%Y%m%d}.npz"
    bp=BASE_DIR/f"{dt:%Y%m%d}.npz"
    fp=TRACK_DIR/f"{dt:%Y%m%d}.npz"

    if not (cache.exists() and bp.exists() and fp.exists()):
        continue

    z=np.load(cache)
    sp=np.hypot(z["u"],z["v"])

    blon,blat=load_xy(bp)
    flon,flat=load_xy(fp)

    rr=qa_fix5.loc[
        qa_fix5["date"]==dt
    ].iloc[0]

    fig,ax=plt.subplots(figsize=(11,7.5))
    im=ax.pcolormesh(
        z["lon"],z["lat"],sp,
        shading="auto"
    )

    ax.plot(
        blon,blat,
        lw=1.1,
        label="original baseline"
    )

    ax.plot(
        flon,flat,
        lw=1.9,
        label="FIX5 event-triggered escape"
    )

    if len(flon):
        ax.scatter(
            flon[-1],flat[-1],
            s=35,marker="x",
            label="FIX5 end"
        )

    ax.set_xlim(120,148)
    ax.set_ylim(20,42)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(
        f"{dt.date()} | "
        f"complete={bool(rr.completed)} "
        f"loops={int(rr.loop_events)} "
        f"escape={int(rr.escape_events)}"
    )
    ax.legend(loc="best")
    fig.colorbar(im,ax=ax,label="speed (m/s)")
    fig.tight_layout()

    fig.savefig(
        FIGURE_DIR/f"{dt:%Y%m%d}_FIX5_compare.png",
        dpi=180
    )
    plt.close(fig)

print("figures ->",FIGURE_DIR)


In [ ]:

# ===== 10. QA通过后自动全量 =====
RUN_FULL = bool(
    cfg.AUTO_RUN_FULL_IF_GATE and
    AUTO_GATE_PASS
)

print("RUN_FULL =",RUN_FULL)

if RUN_FULL:
    available_dates=(
        manifest.loc[
            manifest["cache_status"].isin(["cached","written"]),
            "date"
        ]
        .dropna()
        .sort_values()
        .tolist()
    )

    full_rows=[]

    for dt in tqdm(
        available_dates,
        desc="FIX5 full"
    ):
        dt=pd.Timestamp(dt)
        out=TRACK_DIR/f"{dt:%Y%m%d}.npz"

        try:
            z=np.load(
                CACHE50/f"{dt:%Y%m%d}.npz"
            )
            tr=track_fix5(
                z["lon"],z["lat"],z["u"],z["v"]
            )
            save_track(tr,out)

            m=track_metrics(tr)
            m["date"]=dt
            full_rows.append(m)

        except Exception as e:
            full_rows.append({
                "date":dt,
                "completed":False,
                "loop_events":np.nan,
                "escape_events":np.nan,
                "escape_success":np.nan,
                "n_points":0,
                "stop_reason":"exception",
                "error":repr(e),
            })

    full_report=pd.DataFrame(
        full_rows
    ).sort_values("date")

    full_report.to_csv(
        REPORT_DIR/"full_tracker_report_FIX5.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "full completion:",
        full_report["completed"].mean()
    )
    print(
        "full loop-day:",
        (full_report["loop_events"].fillna(0)>0).mean()
    )
    print(
        "full escape-day:",
        (full_report["escape_events"].fillna(0)>0).mean()
    )
else:
    print(
        "QA Gate未通过或自动全量关闭。"
        "请先检查qa_gate_FIX5.json与manual_qc图。"
    )


In [ ]:

# ===== 11. 全量通过后生成双分支逐日数据 =====
if RUN_FULL:
    from scipy.interpolate import interp1d

    TARGET_LATS_1=np.arange(22.0,25.5+1e-9,1/12)
    TARGET_LONS_2=np.arange(122.0,145.0+1e-9,1/12)

    def branch1_from_track(lons,lats):
        valid=np.isfinite(lons)&np.isfinite(lats)&(lats<=25.7)
        lons=np.asarray(lons)[valid]
        lats=np.asarray(lats)[valid]

        if len(lats)<2:
            return None

        df=(
            pd.DataFrame({"lat":lats,"lon":lons})
            .groupby("lat",as_index=False)
            .lon.mean()
            .sort_values("lat")
        )
        if len(df)<2:
            return None

        f=interp1d(
            df["lat"].values,
            df["lon"].values,
            bounds_error=False,
            fill_value="extrapolate"
        )
        return f(TARGET_LATS_1).astype(np.float32)

    def branch2_from_track(lons,lats):
        valid=np.isfinite(lons)&np.isfinite(lats)
        lons=np.asarray(lons)[valid]
        lats=np.asarray(lats)[valid]

        if len(lons)<2:
            return None

        vals=[]
        for lo in np.sort(np.unique(lons)):
            yy=lats[lons==lo]
            vals.append(
                (
                    lo,
                    np.max(yy) if lo<=125 else np.mean(yy)
                )
            )

        arr=np.asarray(vals,float)
        if len(arr)<2:
            return None

        f=interp1d(
            arr[:,0],arr[:,1],
            bounds_error=False,
            fill_value="extrapolate"
        )
        return f(TARGET_LONS_2).astype(np.float32)

    D=[]; B1=[]; B2=[]

    for r in full_report.itertuples(index=False):
        if not bool(r.completed):
            continue

        p=TRACK_DIR/f"{pd.Timestamp(r.date):%Y%m%d}.npz"
        if not p.exists():
            continue

        z=np.load(p)
        b1=branch1_from_track(z["lon"],z["lat"])
        b2=branch2_from_track(z["lon"],z["lat"])

        if (
            b1 is not None and
            b2 is not None and
            np.isfinite(b1).all() and
            np.isfinite(b2).all()
        ):
            D.append(pd.Timestamp(r.date))
            B1.append(b1)
            B2.append(b2)

    if not D:
        raise RuntimeError("No valid daily dual-branch paths.")

    B1=np.stack(B1)
    B2=np.stack(B2)

    ds_path=DATASET_DIR/"daily_dual_branch_paths_FIX5.npz"

    np.savez_compressed(
        ds_path,
        dates=np.array([
            d.strftime("%Y-%m-%d")
            for d in D
        ]),
        branch1_lon=B1,
        branch1_target_lats=TARGET_LATS_1,
        branch2_lat=B2,
        branch2_target_lons=TARGET_LONS_2,
    )

    meta=pd.DataFrame({
        "date":D,
        "branch1_nan_frac":np.isnan(B1).mean(axis=1),
        "branch2_nan_frac":np.isnan(B2).mean(axis=1),
    })
    meta.to_csv(
        DATASET_DIR/"daily_dual_branch_meta_FIX5.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("dataset:",ds_path)
    print("daily paths:",len(D))


In [ ]:

# ===== 12. Handoff =====
handoff={
    "tracker":"R1_FIX5_EVENT_TRIGGERED_TOPK_ESCAPE",
    "qa":summary,
    "run_full":bool(RUN_FULL),
}

if RUN_FULL:
    handoff.update({
        "full_days":int(len(full_report)),
        "full_completion":float(full_report["completed"].mean()),
        "full_loop_day_rate":float(
            (full_report["loop_events"].fillna(0)>0).mean()
        ),
        "full_escape_day_rate":float(
            (full_report["escape_events"].fillna(0)>0).mean()
        ),
        "full_completed_days":int(full_report["completed"].sum()),
    })

with open(
    REPORT_DIR/"R1_FIX5_HANDOFF.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        handoff,f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    handoff,
    indent=2,
    ensure_ascii=False
))
